### **Hybrid Simulation Using PySD and Parallel Programming**

In [1]:
import psutil
import pysd
import pandas as pd
from joblib import Parallel, delayed

#### Identify number of cores in current device

In [2]:
print("Number of Physical Cores: ", psutil.cpu_count(logical=False))

Number of Physical Cores:  4


#### Pre-Translate (Done once by the main process)

In [3]:
# This creates the .py file version of your model so workers don't have to

model_filename = "Glossi v2.mdl"
py_model_file = pysd.read_vensim(model_filename)

# Get the path of the generated python file

py_path = py_model_file.py_model_file

#### Define a parallelable function

In [4]:
import pandas as pd
def run_single_simulation(num_steps) -> pd.DataFrame:
    """
    Each CPU core will now execute this function, load its own 
    copy of the model, run it, and then clear it from memory.
    """
    # Load the model INSIDE the function so it stays on this core
    local_model = pysd.load(py_path)
    
    # Run the simulation with the specific seed
    result = local_model.run(return_columns=["Dryness Level"], final_time=num_steps-1)
    
    return result["Dryness Level"]

#### Parallelizing the simulation

In [5]:
print(f"Starting single run...")

simulation_results = run_single_simulation(112) #set number of steps here

print(f"Completed single run.")
print(simulation_results.tail())

Starting single run...
Completed single run.
time
107    4.618
108    4.198
109    4.425
110    4.652
111    4.232
Name: Dryness Level, dtype: float64


#### Save results to a CSV file

In [6]:
simulation_results.to_csv("single_run_results.csv")

#### Parse real-world data for Theil's *U*

In [ ]:
real_df = pd.read_csv("glossi-dataset-v1.csv")

# create a column named "avg_dryness" based on "min_dryness" & "max_dryness"
real_df["avg_dryness"] = real_df["min_dryness"] + real_df["max_dryness"] / 2

# isolate "avg_dryness" column
avg_dryness_df = real_df["avg_dryness"]

#### (What to do next?)

### 💡Key Takeaways

Based on the plot, the Stochastic SD model proves to be a highly stable representation of MY hair factors. By parallelizing the sampling of over a thousand runs, it effectively eliminated the noise of human factors and identified a reliable mean value. Additionally, the 99% confidence level still resulted in a tight confidence interval, presenting the model as a high-fidelity decision support system.